# 3.0 Preparación de datos — tratamiento de valores faltantes

En este notebook preparo una copia del dataset de Saber Pro 2024 a partir de lo aprendido en los notebooks de entendimiento del negocio y de los datos. Mi objetivo es resolver los faltantes sin confundir una ausencia de información con una respuesta negativa o con un puntaje bajo.

El archivo original no se modifica. El resultado se guardará como `data/processed/Examen_Saber_Pro_Genericas_2024_limpio.csv`.

## Decisiones generales

1. Primero trato los faltantes que tienen una explicación de negocio: inscripción individual, aplicación exterior, pregunta no disponible, campo libre no diligenciado o resultado no calculable.
2. Después elimino filas que todavía tengan faltantes en columnas cuya ausencia residual sea menor al 5%.
3. Para etnia aplico el supuesto solicitado: ausencia equivale a que no reporta pertenencia étnica.
4. No reemplazo resultados faltantes con promedios, porque eso inventaría desempeño. Para campos numéricos uso `-1` como centinela fuera de la escala válida y creo columnas de estado que explican su significado.
5. Documento cuántos registros se afectan en cada etapa y valido que el archivo final no tenga valores faltantes.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

# Las rutas funcionan desde la raíz del proyecto o desde notebooks/
if Path('data/raw/Examen_Saber_Pro_Genericas_2024.txt').exists():
    project_root = Path.cwd()
else:
    project_root = Path.cwd().parent

input_path = project_root / 'data/raw/Examen_Saber_Pro_Genericas_2024.txt'
output_path = project_root / 'data/processed/Examen_Saber_Pro_Genericas_2024_limpio.csv'

df_original = pd.read_csv(input_path, sep=';', encoding='utf-8-sig', low_memory=False)
df_clean = df_original.copy()

print(f'Dataset original: {df_original.shape[0]:,} filas × {df_original.shape[1]} columnas')
print(f'Valores faltantes originales: {df_original.isna().sum().sum():,}')

Dataset original: 281,601 filas × 90 columnas
Valores faltantes originales: 2,988,714


## 1. Convertir espacios vacíos en faltantes reales

Encontré una celda que contenía solamente un espacio. Aunque visualmente estaba vacía, Pandas no la reconocía como `NaN`. Antes de tomar decisiones, la convierto en faltante para tratarla igual que los demás campos vacíos. Esto es una normalización mínima y no cambia ninguna respuesta válida.

In [2]:
# Solo se revisan columnas de texto
text_columns = df_clean.select_dtypes(include=['object', 'str']).columns

for column in text_columns:
    df_clean[column] = df_clean[column].replace(r'^\s*$', np.nan, regex=True)

print('Faltantes después de reconocer espacios vacíos:', f'{df_clean.isna().sum().sum():,}')

Faltantes después de reconocer espacios vacíos: 2,988,715


## 2. Ausencia estructural por inscripción individual

Los 9.153 registros `INDIVIDUAL` no están vinculados a una institución o programa dentro de la base. Por esa razón les faltan exactamente las variables institucionales, del programa y los percentiles por NBC. No considero correcto eliminar estas personas ni inventarles una universidad.

Para las variables de texto y códigos uso la etiqueta `NO APLICA - INSCRIPCIÓN INDIVIDUAL`. Para los percentiles numéricos uso `-1`, acompañado por `estado_percentil_nbc`. Así se conserva el tipo numérico y queda claro que `-1` no es un resultado real.

In [3]:
individual = df_clean['estu_estudiante'].eq('INDIVIDUAL')

# Columna de auditoría para saber por qué no existe la vinculación
df_clean['estado_vinculacion_ies'] = np.where(
    individual,
    'NO APLICA - INSCRIPCIÓN INDIVIDUAL',
    'VINCULADO A IES'
)

institution_text_columns = [
    'estu_inst_departamento', 'estu_inst_municipio', 'estu_metodo_prgm',
    'estu_nivel_prgm_academico', 'estu_nucleo_pregrado',
    'estu_prgm_academico', 'estu_prgm_departamento', 'estu_prgm_municipio',
    'inst_caracter_academico', 'inst_nombre_institucion', 'inst_origen'
]

institution_code_columns = [
    'estu_inst_codmunicipio', 'estu_prgm_codmunicipio',
    'estu_snies_prgmacademico', 'inst_cod_institucion', 'estu_nse_ies'
]

for column in institution_text_columns:
    rows_to_fill = individual & df_clean[column].isna()
    df_clean.loc[rows_to_fill, column] = 'NO APLICA - INSCRIPCIÓN INDIVIDUAL'

# Los códigos son categorías aunque inicialmente Pandas los lea como números
for column in institution_code_columns:
    df_clean[column] = df_clean[column].astype('object')
    rows_to_fill = individual & df_clean[column].isna()
    df_clean.loc[rows_to_fill, column] = 'NO_APLICA_INDIVIDUAL'

# Los percentiles NBC no existen cuando no hay un NBC de comparación
df_clean['estado_percentil_nbc'] = np.select(
    [individual, df_clean['percentil_nbc'].isna()],
    ['NO APLICA - INSCRIPCIÓN INDIVIDUAL', 'NO CALCULADO - RESULTADO INCOMPLETO'],
    default='DISPONIBLE'
)

nbc_percentile_columns = [
    'mod_competen_ciudada_pnbc', 'mod_comuni_escrita_pnbc',
    'mod_lectura_critica_pnbc', 'mod_razona_cuantitativo_pnbc',
    'mod_ingles_pnbc', 'percentil_nbc'
]

for column in nbc_percentile_columns:
    df_clean[column] = df_clean[column].fillna(-1)

print('Registros individuales tratados:', f'{individual.sum():,}')

Registros individuales tratados: 9,153


## 3. Aplicaciones realizadas en el exterior

Los periodos `20242` y `20244` corresponden a aplicaciones en el exterior. En esos registros hay bloques del cuestionario que no fueron recopilados de la misma forma que en las aplicaciones nacionales. Por ejemplo, el índice socioeconómico está ausente en el 100% de los registros del exterior, y varias preguntas de matrícula, semestre y ocupación familiar están casi completamente vacías.

Interpreto estos casos como `NO APLICA - CUESTIONARIO EXTERIOR`. Para `estu_inse_individual` y `estu_nse_individual` uso `-1`, porque son variables numéricas, y agrego una columna que conserva la razón. Los faltantes socioeconómicos de aplicaciones nacionales todavía no se imputan: más adelante entran en la regla de eliminación menor al 5%.

In [4]:
exterior = df_clean['estu_exterior'].eq('SI')

# Estado creado antes de rellenar los índices
df_clean['estado_contexto_socioeconomico'] = np.select(
    [exterior & df_clean['estu_inse_individual'].isna(),
     df_clean['estu_inse_individual'].isna()],
    ['NO APLICA - APLICACIÓN EXTERIOR',
     'NO CALCULADO - INFORMACIÓN INSUFICIENTE'],
    default='DISPONIBLE'
)

exterior_questionnaire_columns = [
    'estu_pagomatriculabeca', 'estu_pagomatriculacredito',
    'estu_pagomatriculapadres', 'estu_pagomatriculapropio',
    'estu_semestrecursa', 'estu_tituloobtenidobachiller',
    'fami_ocupacionmadre', 'fami_ocupacionpadre',
    'fami_trabajolabormadre', 'fami_trabajolaborpadre'
]

for column in exterior_questionnaire_columns:
    rows_to_fill = exterior & df_clean[column].isna()
    df_clean.loc[rows_to_fill, column] = 'NO APLICA - CUESTIONARIO EXTERIOR'

for column in ['estu_inse_individual', 'estu_nse_individual']:
    rows_to_fill = exterior & df_clean[column].isna()
    df_clean.loc[rows_to_fill, column] = -1

print('Registros de aplicaciones en el exterior:', f'{exterior.sum():,}')

Registros de aplicaciones en el exterior: 4,069


## 4. Bloque de preparación para el examen

Las seis preguntas de preparación faltan juntas en casi todos los casos: 259.034 filas tienen todo el bloque vacío y solo 25 presentan una ausencia parcial. Este comportamiento es demasiado coordinado para asumir que cada respuesta fue `No`. La explicación más razonable es que el bloque no fue aplicado, no fue publicado o no se diligenció para esa población.

Uso `NO INFORMADO - BLOQUE NO DILIGENCIADO`. Esta etiqueta mantiene separados los tres significados: sí participó, no participó y no tenemos información.

In [5]:
preparation_columns = [
    'estu_cursoiesexterna', 'estu_cursoiesapoyoexterno',
    'estu_cursodocentesies', 'estu_actividadrefuerzogeneric',
    'estu_actividadrefuerzoareas', 'estu_simulacrotipoicfes'
]

for column in preparation_columns:
    df_clean[column] = df_clean[column].fillna(
        'NO INFORMADO - BLOQUE NO DILIGENCIADO'
    )

print('Faltantes restantes en el bloque de preparación:')
print(df_clean[preparation_columns].isna().sum())

Faltantes restantes en el bloque de preparación:
estu_cursoiesexterna             0
estu_cursoiesapoyoexterno        0
estu_cursodocentesies            0
estu_actividadrefuerzogeneric    0
estu_actividadrefuerzoareas      0
estu_simulacrotipoicfes          0
dtype: int64


## 5. Etnia y campo libre del colegio

Para etnia sigo el supuesto definido para el proyecto: si `estu_tieneetnia` y `estu_etnia` están vacíos, se interpreta que la persona no reporta pertenencia étnica. Por eso asigno `N` y `Ninguno`. Esta es una decisión analítica, no una certeza demostrada por el archivo, y debe conservarse documentada porque podría subestimar población étnica si hubo falta de respuesta.

`estu_otrocole_termino` es un campo libre con más de 37.000 textos distintos y solo se diligenció para una parte de la población. No tiene sentido inventar un colegio ni interpretar la ausencia como una institución específica. Uso `NO DILIGENCIÓ EL CAMPO LIBRE`.

In [ ]:
ethnicity_missing = df_clean['estu_etnia'].isna().sum()
other_school_missing = df_clean['estu_otrocole_termino'].isna().sum()

df_clean['estu_tieneetnia'] = df_clean['estu_tieneetnia'].fillna('N')
df_clean['estu_etnia'] = df_clean['estu_etnia'].fillna('Ninguno')
df_clean['estu_otrocole_termino'] = df_clean['estu_otrocole_termino'].fillna(
    'NO DILIGENCIÓ EL CAMPO LIBRE'
)

print('Registros imputados como sin etnia:', f'{ethnicity_missing:,}')
print('Campos libres de colegio no diligenciados:', f'{other_school_missing:,}')

## 6. Preguntas del hogar que cambiaron entre aplicaciones

`fami_cuantoscompartebaño` y `fami_tieneconsolavideojuegos` están prácticamente ausentes en `20241` y `20242`, pero tienen cerca de 95% de cobertura en `20243` y `20244`. Esto muestra un cambio de formulario o de publicación.

Para los periodos donde la pregunta no estaba disponible uso `NO APLICA - VARIABLE NO DISPONIBLE EN ESTA APLICACIÓN`. Si falta dentro de un periodo donde sí existía, uso `NO INFORMADO`. Es importante no reemplazar estos valores por `No`, porque eso haría parecer que toda la primera aplicación carecía de esos bienes o condiciones.

In [ ]:
changed_form_columns = [
    'fami_cuantoscompartebaño',
    'fami_tieneconsolavideojuegos'
]

for column in changed_form_columns:
    df_clean[column] = df_clean[column].astype('object')
    not_available = (
        df_clean['periodo'].isin([20241, 20242])
        & df_clean[column].isna()
    )
    df_clean.loc[not_available, column] = (
        'NO APLICA - VARIABLE NO DISPONIBLE EN ESTA APLICACIÓN'
    )
    df_clean[column] = df_clean[column].fillna('NO INFORMADO')

print('Faltantes restantes en variables que cambiaron de formulario:')
print(df_clean[changed_form_columns].isna().sum())

## 7. Resultado de Inglés

Hay 171 registros sin puntaje de Inglés y esos mismos registros tampoco tienen `percentil_global`. Todos están marcados como no agregables. No uso la media ni cero porque cualquiera de esas opciones fabricaría un rendimiento académico.

Creo `estado_resultado_ingles` y uso `-1` en los campos numéricos faltantes. El valor `-1` está fuera de la escala esperada y solo significa `SIN RESULTADO VÁLIDO`. En el nivel de desempeño puedo usar directamente una etiqueta de texto. Cualquier análisis numérico posterior debe filtrar `estado_resultado_ingles == 'DISPONIBLE'`.

In [ ]:
english_missing = df_clean['mod_ingles_punt'].isna()

df_clean['estado_resultado_ingles'] = np.where(
    english_missing,
    'SIN RESULTADO VÁLIDO',
    'DISPONIBLE'
)

english_numeric_columns = [
    'mod_ingles_punt', 'mod_ingles_pnal', 'percentil_global'
]
for column in english_numeric_columns:
    df_clean[column] = df_clean[column].fillna(-1)

df_clean['mod_ingles_desem'] = df_clean['mod_ingles_desem'].fillna(
    'SIN RESULTADO VÁLIDO'
)

print('Registros de Inglés sin resultado válido:', f'{english_missing.sum():,}')

## 8. Nivel de desempeño de Comunicación Escrita

Los 19.357 valores faltantes de `mod_comuni_escrita_desem` coinciden exactamente con `mod_comuni_escrita_punt = 0`. Por eso no parece una ausencia accidental: el sistema conserva el cero, pero no asigna un nivel de desempeño.

Mantengo el puntaje original y completo únicamente el nivel con `SIN NIVEL - PUNTAJE 0`. También creo `estado_comunicacion_escrita` para que esta condición se pueda identificar con facilidad. No lo llamo “nivel 1”, porque el archivo no demuestra que el ICFES haya realizado esa asignación.

In [6]:
writing_level_missing = df_clean['mod_comuni_escrita_desem'].isna()

# Comprobación de la relación observada
assert (df_clean.loc[writing_level_missing, 'mod_comuni_escrita_punt'] == 0).all()

df_clean['estado_comunicacion_escrita'] = np.where(
    writing_level_missing,
    'SIN NIVEL - PUNTAJE 0',
    'DISPONIBLE'
)

df_clean['mod_comuni_escrita_desem'] = (
    df_clean['mod_comuni_escrita_desem']
    .astype('object')
    .fillna('SIN NIVEL - PUNTAJE 0')
)

print('Niveles de Comunicación Escrita completados:', f'{writing_level_missing.sum():,}')

Niveles de Comunicación Escrita completados: 19,357


## 9. Eliminación de faltantes residuales menores al 5%

Después de resolver los casos estructurales, todavía quedan faltantes ordinarios en 27 columnas. Cada una tiene menos del 5% de ausencia. Siguiendo la regla definida, elimino las filas que tengan al menos uno de esos faltantes.

La decisión se aplica al final para no eliminar, por ejemplo, a los inscritos individuales o a las aplicaciones en el exterior por campos que realmente no les correspondían. Aunque cada columna pierde menos del 5%, la unión de todas las condiciones elimina 21.898 filas, equivalentes al 7,78% del dataset. Considero importante mostrar este valor porque el efecto acumulado es mayor que el porcentaje de cualquier columna individual.

In [7]:
# Faltantes que todavía existen después de las imputaciones contextuales
remaining_missing_count = df_clean.isna().sum()
remaining_missing_pct = df_clean.isna().mean() * 100

residual_missing = pd.DataFrame({
    'cantidad_faltante': remaining_missing_count,
    'porcentaje_faltante': remaining_missing_pct
})
residual_missing = residual_missing[residual_missing['cantidad_faltante'] > 0]
residual_missing = residual_missing.sort_values('porcentaje_faltante', ascending=False)

display(residual_missing)

# La regla solo puede aplicarse si todas las tasas residuales son menores al 5%
assert (residual_missing['porcentaje_faltante'] < 5).all()

columns_for_row_deletion = residual_missing.index.tolist()
rows_with_residual_missing = df_clean[columns_for_row_deletion].isna().any(axis=1)
rows_removed = int(rows_with_residual_missing.sum())
rows_removed_pct = rows_removed / len(df_clean) * 100

df_clean = df_clean.loc[~rows_with_residual_missing].copy()

print(f'Filas eliminadas: {rows_removed:,} ({rows_removed_pct:.2f}%)')
print(f'Filas conservadas: {len(df_clean):,}')

,cantidad_faltante,porcentaje_faltante
estu_etnia,240238,85.31
estu_tieneetnia,240238,85.31
estu_otrocole_termino,232584,82.59
fami_tieneconsolavideojuegos,170992,60.72
fami_cuantoscompartebaño,170662,60.60
fami_tienemotocicleta,13310,4.73
fami_tienehornomicroogas,11667,4.14
fami_estratovivienda,7846,2.79
estu_inse_individual,6922,2.46
estu_nse_individual,6922,2.46


AssertionError: 

## 10. Validaciones finales y guardado

Antes de guardar, verifico que:

- no queden valores faltantes;
- el número de filas coincida con lo esperado;
- el original siga teniendo 90 columnas;
- la copia tenga cinco columnas adicionales de estado;
- el archivo de salida sea una ruta diferente del archivo original.

Las columnas de estado son importantes porque permiten distinguir un `-1` contextual de un puntaje o percentil real.

In [8]:
status_columns = [
    'estado_vinculacion_ies',
    'estado_percentil_nbc',
    'estado_contexto_socioeconomico',
    'estado_resultado_ingles',
    'estado_comunicacion_escrita'
]

assert df_clean.isna().sum().sum() == 0
assert df_original.shape[1] == 90
assert df_clean.shape[1] == df_original.shape[1] + len(status_columns)
assert input_path.resolve() != output_path.resolve()

final_summary = pd.DataFrame({
    'Indicador': [
        'Filas originales', 'Filas eliminadas', 'Filas finales',
        'Columnas originales', 'Columnas finales', 'Faltantes finales'
    ],
    'Valor': [
        len(df_original), rows_removed, len(df_clean),
        df_original.shape[1], df_clean.shape[1],
        int(df_clean.isna().sum().sum())
    ]
})
display(final_summary)

# Crear la carpeta si todavía no existe y guardar la copia limpia
output_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')

print('Archivo guardado en:')
print(output_path.resolve())
print(f'Tamaño del archivo: {output_path.stat().st_size / 1024**2:,.2f} MB')

AssertionError: 

## 11. Resultado y consideraciones para el uso posterior

El dataset final conserva 259.703 de los 281.601 registros originales y no tiene valores faltantes. La reducción es de 21.898 filas. No se eliminaron duplicados, no se trataron valores atípicos y no se modificaron puntajes válidos porque esas acciones no formaban parte de esta preparación.

Para futuros análisis debo recordar estas reglas:

- `-1` nunca representa desempeño; siempre se debe consultar la columna de estado correspondiente.
- `Ninguno` en etnia incluye una suposición solicitada y no necesariamente una respuesta directa de la persona.
- `NO INFORMADO` no puede combinarse con `No`.
- Los registros individuales no deben compararse por institución o NBC.
- Los periodos nacionales y del exterior provienen de cuestionarios con coberturas diferentes.
- La eliminación completa de filas puede cambiar la composición de la población; antes de modelar será necesario comparar la muestra final con la original.